# GPU Performance Baseline — Multi-Target Regression

这是一个极简基线模型，使用 **线性回归（Ridge）** 包装在 `MultiOutputRegressor` 中，
同时预测功耗、延迟、能效和吞吐量四个目标。

**A 榜 $R^2$ 得分：**： 0.591480

**关键限制：**
- 未做特征工程（直接使用原始特征）
- 未处理类别特征（简单Label Encoding）
- 未做对数变换
- 未分场景建模

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

print('Libraries loaded.')

In [ ]:
# Load data
train = pd.read_csv('train.csv')
test_a = pd.read_csv('A.csv')
test_b = pd.read_csv('B.csv')

print(f'Train:   {train.shape}')
print(f'Test A:  {test_a.shape}')
print(f'Test B:  {test_b.shape}')

### 查看数据结构

In [ ]:
train.head(3)

In [ ]:
# 目标列
TARGET_COLS = [
    'gpu_power_draw_watts',
    'avg_e2e_latency_seconds',
    'energy_efficiency_tokens_per_joule',
    'throughput_tokens_per_second',
]

# 确认目标列存在
available_targets = [c for c in TARGET_COLS if c in train.columns]
print(f'Targets: {available_targets}')

# 特征列 = 所有列 - 目标列
feature_cols = [c for c in train.columns if c not in TARGET_COLS]
print(f'Features ({len(feature_cols)}): {feature_cols[:10]}...')

### 数据预处理

In [ ]:
# 准备数据
X_train = train[feature_cols].copy()
y_train = train[available_targets].copy()
X_a = test_a[[c for c in feature_cols if c in test_a.columns]].copy()
X_b = test_b[[c for c in feature_cols if c in test_b.columns]].copy()

# 统一列
common_cols = list(set(X_train.columns) & set(X_a.columns) & set(X_b.columns))
X_train = X_train[common_cols]
X_a = X_a[common_cols]
X_b = X_b[common_cols]
print(f'Common feature columns: {len(common_cols)}')

# 丢弃全NaN行
y_train = y_train.dropna(how='all')
valid_idx = y_train.index.intersection(X_train.dropna(how='all').index)
X_train = X_train.loc[valid_idx]
y_train = y_train.loc[valid_idx]
print(f'After cleaning: {len(X_train)} training samples')

In [ ]:
# 分离数值特征和类别特征
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features:   {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)} -> {categorical_cols}')

In [ ]:
# 构建预处理Pipeline
# 数值特征: 中位数填充 + 标准化
# 类别特征: 众数填充 + 标签编码

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', LabelEncoder()),  # Note: ColumnTransformer requires special handling
])

# 对于类别特征，使用 OrdinalEncoder (兼容ColumnTransformer)
from sklearn.preprocessing import OrdinalEncoder
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols),
])

print('Preprocessor ready.')

### 训练模型

使用 Ridge 回归（L2正则化线性模型）包装在 MultiOutputRegressor 中。
这等价于为每个目标独立训练一个线性模型。

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# 构建 Pipeline（直接使用 RandomForestRegressor）
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 训练
print('Training Random Forest model...')
model.fit(X_train, y_train)
print('Training complete!')

In [ ]:
# 训练集上的快速验证
from sklearn.metrics import mean_absolute_error, r2_score

y_pred_train = model.predict(X_train)

for i, target in enumerate(available_targets):
    mae = mean_absolute_error(y_train[target], y_pred_train[:, i])
    r2 = r2_score(y_train[target], y_pred_train[:, i])
    
    # WMAPE
    wmape = np.sum(np.abs(y_train[target] - y_pred_train[:, i])) / np.sum(np.abs(y_train[target])) * 100
    
    print(f'{target}:')
    print(f'  MAE: {mae:.4f}, R²: {r2:.4f}, WMAPE: {wmape:.2f}%')

### 生成预测提交文件

In [ ]:
# 预测A榜
pred_a = model.predict(X_a)
df_pred_a = pd.DataFrame(pred_a, columns=available_targets)
df_pred_a.to_csv('A_predict.csv', index=False)
print(f'A_predict.csv saved: {df_pred_a.shape}')
df_pred_a.head()

In [ ]:
# 预测B榜
pred_b = model.predict(X_b)
df_pred_b = pd.DataFrame(pred_b, columns=available_targets)
df_pred_b.to_csv('B_predict.csv', index=False)
print(f'B_predict.csv saved: {df_pred_b.shape}')
df_pred_b.head()

In [ ]:
# 验证预测文件格式
print('Target ranges in training set:')
for target in available_targets:
    print(f'  {target}: [{y_train[target].min():.2f}, {y_train[target].max():.2f}]')

print('\nPrediction ranges (A):')
for target in available_targets:
    print(f'  {target}: [{df_pred_a[target].min():.2f}, {df_pred_a[target].max():.2f}]')